# Libro del Edificio - Procesador Multimodal para NotebookLM

Este notebook permite procesar interactivamente los documentos del **Libro del Edificio** (PDFs escaneados/nativos e imágenes) y consolida el contenido en archivos Markdown optimizados para NotebookLM.

## 1. Verificación de Configuración y Proveedor Activo

In [ ]:
import os
from pathlib import Path
from dotenv import load_dotenv

load_dotenv()

from config import DOCS_DIR, OUTPUT_DIR, PROVEEDOR, MODEL_NAME, API_KEY, ALLOWED_EXTENSIONS, IGNORED_EXTENSIONS, get_group_filename
from processor import CheckpointManager, get_vision_client, DocumentProcessor

print(f"Directorio Origen : {DOCS_DIR}")
print(f"Directorio Destino: {OUTPUT_DIR}")
print(f"Proveedor Activo  : {PROVEEDOR}")
print(f"Modelo Seleccionado: {MODEL_NAME}")

## 2. Inspección del Estructura de Documentos

In [ ]:
files_to_process = []
for root, _, files in os.walk(DOCS_DIR):
    for f in files:
        fp = Path(root) / f
        ext = fp.suffix.lower()
        if ext in ALLOWED_EXTENSIONS and ext not in IGNORED_EXTENSIONS:
            rel = fp.relative_to(DOCS_DIR)
            group = get_group_filename(rel)
            files_to_process.append((rel, group))

print(f"Total archivos detectados: {len(files_to_process)}\n")
for rel, group in files_to_process[:10]:
    print(f"- Documento: {rel}  ==>  Destino: {group}")
if len(files_to_process) > 10:
    print(f"... y {len(files_to_process) - 10} archivos más.")

## 3. Prueba de Extracción en un Documento

In [ ]:
if files_to_process:
    sample_rel, sample_group = files_to_process[0]
    sample_file = DOCS_DIR / sample_rel
    print(f"Procesando prueba: {sample_rel}")
    
    checkpoint_mgr = CheckpointManager()
    vision_client = get_vision_client(provider=PROVEEDOR, api_key=API_KEY, model_name=MODEL_NAME)
    doc_processor = DocumentProcessor(vision_client, checkpoint_mgr)
    
    target_test_md = OUTPUT_DIR / f"TEST_{sample_group}"
    completed = doc_processor.process_file_incremental(sample_file, str(sample_rel), target_test_md, sample_group)
    print(f"Procesamiento de prueba completado: {completed}")

## 4. Ejecución del Pipeline Completo de Consolidación

In [ ]:
from main import run_pipeline

run_pipeline()

## 5. Resumen de Archivos Markdown Consolidados Generados

In [ ]:
if OUTPUT_DIR.exists():
    output_files = list(OUTPUT_DIR.glob("*.md"))
    print(f"Se generaron {len(output_files)} archivos consolidados en {OUTPUT_DIR}:\n")
    for out_f in output_files:
        size_kb = out_f.stat().st_size / 1024
        print(f"- {out_f.name} ({size_kb:.1f} KB)")
else:
    print("Aún no se ha ejecutado la consolidación.")